In [49]:
import pandas as pd
import sys
import spacy
import re
import numpy as np

In [3]:
#read in file
file = pd.read_csv("bios.tsv.gz", sep="\t", compression="gzip", index_col = 0)
bio_df = file.set_index("title")

In [11]:
bio_df.head()

,gender,clean_bio
title,,
David Campese,male,"David Campese was born on 21 October 1962, in ..."
Frank Sinatra,male,Francis Albert Sinatra was born on December 12...
Elvis Presley,male,"Elvis Aaron Presley was born on January 8, 193..."
Akira Kurosawa,male,"Kurosawa was born on March 23, 1910, in Ōimach..."
Adolphe Thiers,male,"Adolphe Thiers was born on 15 April 1797, duri..."


In [22]:
NOUN_POS = {'NOUN','PRON','PROPN'}
VERB_POS = {'AUX','VERB'}

def get_root(sent):
    root = sent.root
    if root.pos_ in VERB_POS:
        return root
    else:
        return None

SUBJ_DEPS = {'nsubj','nsubjpass', 'csubj', 'csubjpass'}

def get_subj(sent):
    root = get_root(sent)
    if root is None: return None

    for child in root.children:
        if child.dep_ in SUBJ_DEPS:
            return child

OBJ_DEPS = {'obj', 'dobj', 'iobj',  'dative', 'attr'}

def get_obj(sent):
    root = get_root(sent)
    if root is None: return None

    for child in root.children:
        if child.dep_ in OBJ_DEPS:
            return child

#return list of (prep, pobj) tuples 
def get_pobjs(sent):
    pobjs = []
    root = get_root(sent)
    if root is None: return []
    
    for child in root.children:
        if child.dep_ == "prep":
            for prep_child in child.children:
                if prep_child.dep_ == "pobj":
                    pobjs.append((child, prep_child))
    return pobjs

In [23]:
eg_tiny = '''Bowie made music.'''
eg_short = '''Bowie was born on 8 January 1947 in Brixton.'''
eg_mid = '''When he was a young adult, David Bowie moved to New York City in 1974.'''
eg_long = '''Dressed in a striking costume, Bowie launched his Ziggy Stardust stage show in Kingston on 10 February 1972.'''

In [24]:
def get_spacy_sent(text):
    return list(nlp(text).sents)[0]

In [25]:
nlp = spacy.load('en_core_web_sm')

In [26]:
spacy_eg_tiny = get_spacy_sent(eg_tiny)
spacy_eg_short = get_spacy_sent(eg_short)
spacy_eg_mid = get_spacy_sent(eg_mid)
spacy_eg_long = get_spacy_sent(eg_long)

spacy_egs = [spacy_eg_tiny,spacy_eg_short, spacy_eg_mid, spacy_eg_long]

In [27]:
for i, sent in enumerate(spacy_egs):
    print('[%d]' % i, get_subj(sent), '|', get_root(sent))
    obj = get_obj(sent)
    if obj:
        print('\t\t', obj)
    for prep, pobj in get_pobjs(sent):
        print('\t\t', '(%s)' % prep, pobj)

[0] Bowie | made
		 music
[1] Bowie | born
		 (on) January
		 (in) Brixton
[2] Bowie | moved
		 (to) City
		 (in) 1974
[3] Bowie | launched
		 show
		 (in) Kingston
		 (on) February


In [31]:
#get noun phrases for token
def convert_token_to_span(tok):
    return tok.doc[tok.i:tok.i+1]

def get_noun_chunk(tok):
    if tok is None: return None
    sentence = tok.sent

    #check if token is in a noun chunk
    for chunk in sentence.noun_chunks:
        if chunk.start <= tok.i < chunk.end:
            return chunk

    #if not in noun chunk, return token as span
    return convert_token_to_span(tok)

In [32]:
for i, sent in enumerate(spacy_egs):
    print('[%d]' % i, get_noun_chunk(get_subj(sent)), '|', get_root(sent))
    obj = get_obj(sent)
    if obj:
        print('\t\t', get_noun_chunk(obj))
    for prep, pobj in get_pobjs(sent):
        print('\t\t', '(%s)' % prep, get_noun_chunk(pobj))

[0] Bowie | made
		 music
[1] Bowie | born
		 (on) 8 January
		 (in) Brixton
[2] David Bowie | moved
		 (to) New York City
		 (in) 1974
[3] Bowie | launched
		 his Ziggy Stardust stage show
		 (in) Kingston
		 (on) 10 February


In [33]:
#get date. if multiple dates, return first.
#date contains 4 consecutive digits
def get_date(sent):
    pattern = re.compile(r"\d{4}")
    for ent in sent.ents:
        #check if label id DATE
        if ent.label_ == "DATE":
            #check if 4 consecutive digits
            text = ent.text
    
            matches = pattern.findall(text)
            #if found, return entity. Ensure return first match
            if len(matches) == 1:
                return ent

    return None

In [36]:
for i, sent in enumerate(spacy_egs):
    print('[%d]' % i, get_date(sent))

[0] None
[1] 8 January 1947
[2] 1974
[3] 10 February 1972


Apply functions to the data

In [50]:
from spacy.tokens import DocBin

spacy_file = "bios.spacy"
docs = list(DocBin().from_disk(spacy_file).get_docs(nlp.vocab))

#keyed by person name
docs = {bio_df.index[i]: docs[i] for i in np.arange(len(bio_df))}

In [40]:
def get_name_tokens(title, gender):
    if gender == "male":
        name_tokens = ["he"]
    elif gender == 'female':
        name_tokens = ['she']
    name_tokens += [x.lower() for x in title.replace('-',' ').split()] 
    return set(name_tokens)

In [41]:
name_token_dict = {}
for title, gender in bio_df.gender.items():
    name_token_dict[title] = get_name_tokens(title,gender)

In [42]:
def is_mention(tok, person):
    if tok is None: return False
    return tok.text.lower() in name_token_dict[person]

In [46]:
def get_timeline_sents(person):
    doc = docs[person] # get the spaCy version of the person's biography
    valid_sents = []

    for sentence in doc.sents:
        #check for subj
        subj = get_subj(sentence)
        if is_mention(subj, person) == False:
            continue

        #check if sentence has date
        if get_date(sentence) == None:
            continue

        valid_sents.append(sentence)
            
    return valid_sents

In [51]:
example_people = ['David Bowie']
timeline_sent_dict = {person: get_timeline_sents(person) for person in example_people}

In [52]:
for person in example_people:
    print(person)
    for i, sent in enumerate(timeline_sent_dict[person]):
        date, root, subj = get_date(sent), get_root(sent), get_subj(sent)
        print('[%d]' % i, date, '\n\t', get_noun_chunk(subj), '|', root)
        obj = get_obj(sent)
        if obj:
            print('\t\t', get_noun_chunk(obj))
        for prep, pobj in get_pobjs(sent):
            print('\t\t', '(%s)' % prep, get_noun_chunk(pobj))
    print('\n\n\n========\n\n\n')

David Bowie
[0] 8 January 1947 
	 Bowie | born
		 David Robert Jones
		 (on) 8 January
[1] 1953 
	 Bowie | moved
		 (In) 1953
		 (with) his family
		 (to) Bromley
[2] 1962 
	 He | received
		 a serious injury
		 (at) school
		 (in) 1962
[3] 1967 
	 Bowie | met
		 dancer Lindsay Kemp
		 (in) 1967
[4] 1972 
	 He | commented
		 (in) 1972
[5] early 1969 
	 he | wore
		 a T-shirt
[6] March 1969 
	 he | undertook
		 a short tour
		 (In) February
[7] April 1969 
	 
Bowie | met
		 Angela Barnett
		 (in) April
[8] 2002 
	 Bowie | cover
		 ''Heathen
[9] 10 February 1972 
	 Bowie | launched
		 his Ziggy Stardust stage show
		 (with) the Spiders
		 (from) Mars
		 (at) the Toby Jug pub
		 (on) 10 February
[10] 1972 
	 Bowie | contributed
		 backing vocals
[11] 3 July 1973 
	 Bowie | toured
[12] 1974 
	 Bowie | moved
		 (to) the US
		 (in) 1974
[13] between June and December 1974 
	 Bowie | launched
		 the Diamond Dogs Tour
[14] 1975 
	 Bowie | fired
		 his manager
		 (In) 1975
		 (in) a move
[15] 1

Identify places in biographies

In [55]:
def get_locations(person):
    places = []
    timeline_sents = timeline_sent_dict[person]
    for sent in timeline_sents:
        for ent in sent.ents:
            if ent.label_ == "GPE":
                places.append(ent.text)
            

    return set(places)

In [56]:
for loc in get_locations('David Bowie'):
    print(loc)

Gemini
UK
America
Manhattan
Switzerland
Norway
Toronto
London
Best Rock Performance
Lausanne
New York
Bowie
England
Los Angeles
Kingston
Brixton
New York City
US
Chile
